# NEXUS CYP3A4 Cache + Train on Colab

Run this notebook top-to-bottom, or stop after the cache build and resume training later.

What it does:
- builds an offline CYP3A4 physics cache with `scripts/colab_precompute_cyp3a4_physics_cache.py`
- saves the cache to Drive when available
- trains with cached physics in `hybrid` or `cached` mode using `scripts/colab_train_cyp3a4.py`
- keeps the broad analogical memory bank and exact-query retrieval masking


In [ ]:
# Cell 1: clone/pull repo and install package
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull --ff-only', shell=True, check=True)

subprocess.run(f'pip -q install -e {REPO_DIR}', shell=True, check=True)
print('Repo ready:', REPO_DIR)


In [ ]:
# Cell 2: optional Google Drive mount for persistent cache/checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except Exception as e:
    print('Drive mount skipped:', e)


In [ ]:
# Cell 3: cache-build config
import os

# Cache presets:
#   balanced  - 64 CYP3A4 molecules, trimmed full-physics cache build
#   full_3a4  - full CYP3A4 dataset cache build
os.environ['NEXUS_COLAB_CACHE_PRESET'] = 'full_3a4'

# Optional overrides
os.environ['NEXUS_COLAB_TARGET_ISOFORM'] = '3A4'
os.environ['NEXUS_COLAB_GPU_PROFILE'] = 'ultra_vram'

# Default cache path goes to Drive if mounted; override here if needed.
# os.environ['NEXUS_COLAB_PHYSICS_CACHE_PATH'] = '/content/drive/MyDrive/nexus_cyp3a4_physics_cache.pt'

print('Cache preset:', os.environ['NEXUS_COLAB_CACHE_PRESET'])
print('Target isoform:', os.environ['NEXUS_COLAB_TARGET_ISOFORM'])


In [ ]:
# Cell 4: build / resume offline physics cache
exec(open('/content/enzyme_Software/scripts/colab_precompute_cyp3a4_physics_cache.py').read())


In [ ]:
# Cell 5: training config
import os

# Training presets:
#   fast          - 5 epochs, 32 molecules — smoke-test / debug
#   balanced      - 10 epochs, all molecules — default iteration preset
#   full_3a4      - 15 epochs, all molecules — Colab A100/H100 target
#   full_3a4_a100 - 30 epochs, full physics (dynamics_steps=2, scan_pts=32) — cloud A100/H100
#   rtx6k_2h      - 25 epochs, curriculum physics — RTX 6000 Ada / 95 GB target
os.environ['NEXUS_COLAB_RUN_PRESET'] = 'full_3a4'

# Cell-level overrides (leave blank to use preset defaults):
# Increase epochs beyond preset default for longer runs.
# os.environ['NEXUS_COLAB_EPOCHS'] = '20'

# Remove the sample cap (0 = all molecules — REQUIRED for real accuracy).
os.environ['NEXUS_COLAB_MAX_SAMPLES'] = '0'

# Cache behavior:
#   hybrid - use cached physics when available, fall back to live dynamics
#   cached - require cache entries; fail on a miss
os.environ['NEXUS_COLAB_USE_PHYSICS_CACHE'] = '1'
os.environ['NEXUS_COLAB_PHYSICS_CACHE_MODE'] = 'hybrid'

# DAG loss warm-up: ramp DAG contribution from 0 → full over this many steps.
# Prevents the large acyclicity penalty from drowning the SoM ranking signal
# during early training.  Each preset sets a sensible default; override here
# only if you know what you're doing.
# os.environ['NEXUS_COLAB_DAG_WARMUP_STEPS'] = '800'

# Optional strict cache path override
# os.environ['NEXUS_COLAB_PHYSICS_CACHE_PATH'] = '/content/drive/MyDrive/nexus_cyp3a4_physics_cache.pt'

# Optional: start a fresh run instead of resuming the previous checkpoint.
# CKPT = '/content/drive/MyDrive/nexus_colab_checkpoint.pt'
# if os.path.exists(CKPT):
#     os.remove(CKPT)

print('Training preset:', os.environ['NEXUS_COLAB_RUN_PRESET'])
print('Cache mode:', os.environ['NEXUS_COLAB_PHYSICS_CACHE_MODE'])
print('Max samples:', os.environ.get('NEXUS_COLAB_MAX_SAMPLES', '(preset default)'))


In [ ]:
# Cell 6: train with cached physics
exec(open('/content/enzyme_Software/scripts/colab_train_cyp3a4.py').read())
